# OCR Stiker Obat - End to End v2 (YOLO 2 kelas + PaddleOCR)
Pipeline: **YOLO 2 kelas (ambil `entete`) -> orientation Paddle (0/90/180/270) -> rotate + enhance -> PaddleOCR -> teks**

Beda dari versi sebelumnya:
- **YOLO** memakai setelan seperti punya temanmu: 2 kelas (`entete` dan `vin`), augmentasi rotasi (`degrees=180`, `flipud=0.5`), `CONF=0.1`, hanya kotak `entete` yang dipakai. Model disimpan sebagai `models/yolo_2cls.pt` (yang lama tidak ditimpa).
- **Labeling diulang** ke file baru `work/labels_v2.csv` (yang lama `work/labels.csv` tidak disentuh). GUI bisa menampilkan **draf dari PaddleOCR** supaya kamu tinggal memeriksa dan memperbaiki.
- Sisa training PyTorch (ResNet orientation) dan scorecard lama dibuang. Orientation tetap Paddle + kalibrasi.

| Bagian | Isi | Catatan |
|---|---|---|
| 0 | Setup & fungsi bantu | selalu jalan |
| 1 | Siapkan data (YOLO 2 kelas + crop + `labels_v2.csv`) | sekali saja |
| 2 | **Labeling manual** (draf Paddle opsional) | `rot` untuk kalibrasi orientation, `text` untuk ukur CER |
| 3 | Train YOLO | dilewati kalau `models/yolo_2cls.pt` sudah ada |
| 4 | Kalibrasi orientation | tidak dilatih, hanya memetakan label Paddle ke rotasi koreksi |
| 5 | Ukur PaddleOCR (CER) | memilih `ENHANCE` |
| 6 | Inference | menghasilkan `submission_v2.csv` dan `debug_v2.csv` |

Struktur folder: `data/train`, `data/test`, `data/sample_submission.csv`. Folder `work/` dan `models/` dibuat otomatis.

## 0. Setup & fungsi bantu

In [ ]:
# pip install ultralytics rapidfuzz paddlepaddle "paddleocr" tqdm     # jalankan sekali kalau belum ada
import json, pickle, random, re, shutil
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
from tqdm import tqdm
from ultralytics import YOLO

DATA, WORK, MODELS = Path("data"), Path("work"), Path("models")
CROPS, CSV = WORK / "crops", WORK / "labels_v2.csv"       # label BARU; work/labels.csv yang lama tidak disentuh
YOLO_DIR = WORK / "yolo2"
for d in [CROPS, MODELS, WORK / "test_crops_v2"]:
    d.mkdir(parents=True, exist_ok=True)

SEED         = 42
CLASS_NAMES  = ["entete", "vin"]     # urutan = nomor kelas YOLO (entete = 0, vin = 1)
COCO_TO_YOLO = {1: 0, 2: 1}          # COCO: 0='Images' (dummy), 1='entete', 2='vin'
PAD          = 4                     # padding crop (px)
IMGSZ        = 416                   # ukuran input YOLO
CONF         = 0.1                   # ambang confidence YOLO (sama seperti punya temanmu)
UPSCALE      = 3                     # crop diperbesar dulu karena tulisannya kecil

random.seed(SEED); np.random.seed(SEED)

In [ ]:
# ---------- Fungsi bantu ----------
SKIP = "__SKIP__"      # penanda crop yang dilewati saat labeling

def clean_text(text):
    """Huruf besar, buang semua selain A-Z 0-9 / -  (spasi, ':', '.', dll ikut dibuang)."""
    return re.sub(r"[^A-Z0-9/\-]", "", str(text).upper())

# KONVENSI ROTASI: rot = derajat putar SEARAH JARUM JAM agar teks tegak. Kelas orientation = rot // 90.
_CV_ROT = {90: cv2.ROTATE_90_CLOCKWISE, 180: cv2.ROTATE_180, 270: cv2.ROTATE_90_COUNTERCLOCKWISE}

def rotate_cw(img, rot):
    return img if rot % 360 == 0 else cv2.rotate(img, _CV_ROT[rot % 360])

def needed_rot(applied):
    """Gambar tegak yang diputar `applied` derajat cw butuh berapa derajat cw untuk ditegakkan lagi."""
    return (360 - applied) % 360

ENHANCE = True   # CLAHE + sharpen tipis sebelum OCR. Nilainya dipilih ulang di bagian 5.

def enhance(img, on=None):
    """Perbaiki kontras (CLAHE di kanal L) + sharpen tipis. `on=None` -> ikut flag ENHANCE."""
    if not (ENHANCE if on is None else on):
        return img
    l, a, b = cv2.split(cv2.cvtColor(img, cv2.COLOR_BGR2LAB))
    l = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(4, 4)).apply(l)
    img = cv2.cvtColor(cv2.merge([l, a, b]), cv2.COLOR_LAB2BGR)
    return cv2.addWeighted(img, 1.4, cv2.GaussianBlur(img, (0, 0), 1.0), -0.4, 0)

def merge_lines(results):
    """Urutkan hasil PaddleOCR: baris atas -> bawah, dalam satu baris kiri -> kanan.
    results = [(bbox 4 titik, teks, confidence), ...]"""
    items = []
    for box, text, conf in results:
        ys, xs = [p[1] for p in box], [p[0] for p in box]
        items.append({"cy": sum(ys) / 4, "h": max(ys) - min(ys), "x": min(xs), "text": text})
    items.sort(key=lambda d: d["cy"])
    lines = []
    for it in items:
        if lines and abs(it["cy"] - lines[-1][0]["cy"]) < 0.6 * max(it["h"], 1):
            lines[-1].append(it)                       # masih satu baris
        else:
            lines.append([it])
    return [it for line in lines for it in sorted(line, key=lambda d: d["x"])]

_paddle = None

def get_paddle():
    """Muat PaddleOCR sekali saja (butuh internet sekali untuk mengunduh modelnya)."""
    global _paddle
    if _paddle is None:
        from paddleocr import PaddleOCR
        _paddle = PaddleOCR(lang="en", use_doc_orientation_classify=False,
                            use_doc_unwarping=False, use_textline_orientation=False, enable_mkldnn=False)
    return _paddle

def _field(res, key):
    """Ambil field hasil PaddleOCR (versi berbeda membungkusnya sedikit berbeda)."""
    try:
        return res[key]
    except (KeyError, TypeError):
        return res.json["res"][key]

def paddle_read(img_upright, use_enhance=None, ocr=None):
    """Baca crop yang SUDAH tegak dengan PaddleOCR -> teks bersih (A-Z 0-9 / -)."""
    ocr = ocr or get_paddle()
    img = enhance(img_upright, use_enhance)
    big = cv2.resize(img, None, fx=UPSCALE, fy=UPSCALE, interpolation=cv2.INTER_CUBIC)
    items = []
    for res in ocr.predict(big):
        items += [(poly, text, 1.0) for poly, text in zip(_field(res, "rec_polys"), _field(res, "rec_texts"))]
    return clean_text("".join(it["text"] for it in merge_lines(items)))

def load_labeled():
    """Baris labels_v2.csv yang sudah terisi (tanpa yang di-skip)."""
    df = pd.read_csv(CSV, dtype=str, keep_default_na=False)
    df = df[(df["text"] != "") & (df["text"] != SKIP)].copy()
    df["rot"] = df["rot"].astype(int)
    return df

def load_upright(file, rot):
    """Crop yang sudah diputar sehingga teks tegak."""
    return rotate_cw(cv2.imread(str(CROPS / file)), rot)

def cer(preds, refs):
    """Character Error Rate = total jarak Levenshtein / total panjang referensi."""
    from rapidfuzz.distance import Levenshtein
    errors = sum(Levenshtein.distance(p, r) for p, r in zip(preds, refs))
    return errors / max(1, sum(len(r) for r in refs))

_ori = None

def get_ori():
    """Muat classifier orientasi bawaan PaddleOCR sekali saja (0/90/180/270)."""
    global _ori
    if _ori is None:
        from paddleocr import DocImgOrientationClassification
        _ori = DocImgOrientationClassification(model_name="PP-LCNet_x1_0_doc_ori")
    return _ori

def ori_label(img_bgr):
    """Label mentah dari Paddle (0/90/180/270). Arti arahnya dikalibrasi di bagian 4 -> ORI_MAP."""
    res = next(iter(get_ori().predict(img_bgr, batch_size=1)))
    return int(_field(res, "label_names")[0])

ORI_MAP = {0: 0, 90: 90, 180: 180, 270: 270}   # label Paddle -> derajat cw koreksi; ditimpa hasil kalibrasi

def fix_rot(img_bgr):
    """Putar crop agar tegak memakai orientation Paddle + ORI_MAP."""
    return rotate_cw(img_bgr, ORI_MAP.get(ori_label(img_bgr), 0))

## 1. Siapkan data
Membuat (a) dataset YOLO **dua kelas** (`entete` = 0, `vin` = 1) di `work/yolo2/`, (b) crop `entete` dari kotak train + `labels_v2.csv` kosong untuk diisi.
**Aman dijalankan ulang:** `labels_v2.csv` tidak akan ditimpa kalau sudah ada (supaya label-mu tidak hilang).

In [ ]:
coco = json.load(open(DATA / "train" / "_annotations.coco.json"))

# Perbaiki 1 annotation rusak (key '' dan 'catimage_idegory_id')
for ann in coco["annotations"]:
    if "image_id" not in ann:
        ann["image_id"], ann["category_id"] = ann.pop(""), ann.pop("catimage_idegory_id")

images = {img["id"]: img for img in coco["images"]}
anns = [a for a in coco["annotations"] if a["category_id"] in COCO_TO_YOLO]     # buang kelas dummy 'Images'
entete_anns = [a for a in anns if COCO_TO_YOLO[a["category_id"]] == 0]
print("gambar:", len(images), "| box entete:", len(entete_anns), "| box vin:", len(anns) - len(entete_anns))

# Bagi train/val per GAMBAR (bukan per box) supaya tidak bocor
ids = sorted(images); random.Random(SEED).shuffle(ids)
val_ids = set(ids[: int(0.2 * len(ids))])

anns_by_img = {}
for a in anns:
    anns_by_img.setdefault(a["image_id"], []).append(a)

# (a) dataset YOLO 2 kelas, koordinat dinormalisasi (dibuat ulang dari nol setiap dijalankan)
shutil.rmtree(YOLO_DIR, ignore_errors=True)
for part in ("train", "val"):
    (YOLO_DIR / "images" / part).mkdir(parents=True)
    (YOLO_DIR / "labels" / part).mkdir(parents=True)

for img_id, img in images.items():
    split = "val" if img_id in val_ids else "train"
    shutil.copy(DATA / "train" / img["file_name"], YOLO_DIR / "images" / split / img["file_name"])
    W, H = img["width"], img["height"]
    lines = []
    for a in anns_by_img.get(img_id, []):
        x, y, w, h = a["bbox"]
        cls = COCO_TO_YOLO[a["category_id"]]
        lines.append(f"{cls} {(x + w/2)/W:.6f} {(y + h/2)/H:.6f} {w/W:.6f} {h/H:.6f}")
    (YOLO_DIR / "labels" / split / (Path(img["file_name"]).stem + ".txt")).write_text("\n".join(lines))

(YOLO_DIR / "data.yaml").write_text(
    f"path: {YOLO_DIR.resolve().as_posix()}\ntrain: images/train\nval: images/val\nnames:\n  0: entete\n  1: vin\n")

# (b) crop entete + template labels_v2.csv (HANYA kalau belum ada)
if CSV.exists():
    print(CSV.name, "sudah ada -> crop & label TIDAK ditimpa.")
else:
    rows = []
    for a in entete_anns:
        img = images[a["image_id"]]
        im = cv2.imread(str(DATA / "train" / img["file_name"]))
        x, y, w, h = [int(v) for v in a["bbox"]]
        crop = im[max(0, y - PAD): y + h + PAD, max(0, x - PAD): x + w + PAD]
        name = f"{Path(img['file_name']).stem}_{a['id']}.jpg"
        cv2.imwrite(str(CROPS / name), crop)
        rows.append({"file": name, "split": "val" if a["image_id"] in val_ids else "train", "rot": "", "text": ""})
    pd.DataFrame(rows).to_csv(CSV, index=False)
    print("crop tersimpan:", len(rows), "->", CSV.name)

## 2. Labeling manual
**Format label (asumsi, belum dikonfirmasi panitia):** teks penuh persis yang tercetak di entete, dari atas ke bawah, termasuk kata kunci, tanpa spasi. Contoh: `LOTGI005FAB10/19PER10/21`. Ini sama dengan bentuk keluaran Paddle dan keluaran notebook peserta lain, jadi labelnya langsung sebanding dengan hasil OCR.

**2a. Draf otomatis (opsional, lambat).** Untuk `DRAFT_N` crop pertama (urutan sama dengan GUI), PaddleOCR menebak rotasi (mencoba 90 dan 270, karena entete selalu tegak/vertikal) dan teksnya. Hasilnya disimpan di `work/drafts.csv`, bisa dilanjutkan kalau terhenti. Lewati cell ini kalau mau mengetik dari nol, GUI tetap jalan.

In [ ]:
DRAFTS = WORK / "drafts.csv"
DRAFT_N = 50                       # berapa crop pertama yang dibuatkan draf (kira-kira 5-8 detik per crop di CPU)
KEYWORDS = r"LOT|FAB|EXP|PER|DDF|DDP|DE"

def label_order(df):
    """Urutan crop di GUI (diacak supaya train & val sama-sama terisi). Dipakai juga oleh pembuat draf."""
    order = list(df.index); random.Random(SEED).shuffle(order)
    return order

def draft_for(file):
    """Coba rotasi 90 dan 270, pilih yang bacaannya paling mirip stiker (banyak kata kunci, lalu paling panjang)."""
    img = cv2.imread(str(CROPS / file))
    best = None
    for rot in (90, 270):
        text = paddle_read(rotate_cw(img, rot))
        score = 100 * len(re.findall(KEYWORDS, text)) + len(text)
        if best is None or score > best[0]:
            best = (score, rot, text)
    return best[1], best[2]

def make_drafts(n=DRAFT_N):
    df = pd.read_csv(CSV, dtype=str, keep_default_na=False)
    rows = pd.read_csv(DRAFTS, dtype=str, keep_default_na=False).to_dict("records") if DRAFTS.exists() else []
    have = {r["file"] for r in rows}
    todo = [df.at[i, "file"] for i in label_order(df)[:n]
            if df.at[i, "text"] == "" and df.at[i, "file"] not in have]
    for f in tqdm(todo, desc="draf Paddle"):
        rot, text = draft_for(f)
        rows.append({"file": f, "rot": str(rot), "text": text})
        if len(rows) % 10 == 0:
            pd.DataFrame(rows).to_csv(DRAFTS, index=False)
    pd.DataFrame(rows).to_csv(DRAFTS, index=False)
    print("draf tersedia:", len(rows))

make_drafts(DRAFT_N)

**2b. GUI labeling.** Satu jendela (Tkinter). Per crop:
1. Kalau ada draf, tampil teks tebakan Paddle dan gambar sudah diputar sesuai tebakannya. **Periksa dua hal**: gambar benar-benar tegak (kalau belum, **Ctrl+R** putar 90 derajat searah jarum jam), dan teksnya benar huruf per huruf (koreksi di kotak teks). Draf Paddle sering salah di `0/O`, `1/I`, dan urutan baris.
2. Tekan **Enter** untuk menyimpan (mengisi `rot` dan `text`).
3. Tidak terbaca sama sekali: **Lewati**.

Tersimpan otomatis tiap Enter; boleh ditutup dan dibuka lagi. Target awal **>= 100 crop** (minimal 20 untuk kalibrasi orientation). Jangan menekan Enter beruntun tanpa memeriksa: label yang diterima buta membuat angka CER dan kalibrasi orientation terlihat lebih bagus dari kenyataannya.

In [ ]:
def run_label_tool():
    import tkinter as tk
    from tkinter import messagebox
    from PIL import Image, ImageTk

    VIEW_W, VIEW_H = 900, 420
    df = pd.read_csv(CSV, dtype=str, keep_default_na=False)
    order = label_order(df)
    drafts = {}
    if DRAFTS.exists():
        d = pd.read_csv(DRAFTS, dtype=str, keep_default_na=False)
        drafts = {f: (int(r), t) for f, r, t in zip(d["file"], d["rot"], d["text"])}
    start = next((p for p, i in enumerate(order) if df.at[i, "text"] == ""), 0)

    root = tk.Tk(); root.title("Label entete")
    state = {"pos": start, "rot": 0, "photo": None}

    canvas = tk.Label(root, bg="#222"); canvas.pack(padx=10, pady=10)
    info = tk.Label(root, font=("Arial", 11)); info.pack()
    draft_note = tk.Label(root, font=("Arial", 11, "bold"), fg="#c60"); draft_note.pack()
    preview = tk.Label(root, font=("Consolas", 12), fg="#0a6"); preview.pack()
    text_var = tk.StringVar()
    entry = tk.Entry(root, textvariable=text_var, font=("Consolas", 16), width=60); entry.pack(padx=10, pady=6)
    text_var.trace_add("write", lambda *_: preview.config(text="disimpan sebagai: " + clean_text(text_var.get())))

    def show():
        row = df.loc[order[state["pos"]]]
        im = Image.open(CROPS / row["file"]).convert("RGB").rotate(-state["rot"], expand=True)  # -rot = searah jarum jam
        s = min(VIEW_W / im.width, VIEW_H / im.height)
        im = im.resize((int(im.width * s), int(im.height * s)), Image.LANCZOS)
        state["photo"] = ImageTk.PhotoImage(im); canvas.config(image=state["photo"])
        done = int((df["text"] != "").sum())
        info.config(text=f"[{state['pos'] + 1}/{len(order)}]  {row['file']}  |  rot = {state['rot']}  |  terlabel: {done}")

    def goto(pos):
        state["pos"] = max(0, min(len(order) - 1, pos))
        row = df.loc[order[state["pos"]]]
        is_draft = row["text"] == "" and row["file"] in drafts          # belum dilabel, ada draf dari Paddle
        if is_draft:
            state["rot"], text = drafts[row["file"]]
        else:
            state["rot"] = int(row["rot"]) if row["rot"] != "" else 0
            text = "" if row["text"] in ("", SKIP) else row["text"]
        text_var.set(text)
        draft_note.config(text="DRAF PaddleOCR: cek rotasi dan teks huruf per huruf, lalu Enter" if is_draft else "")
        show(); entry.focus_set()

    def rotate(_=None):
        state["rot"] = (state["rot"] + 90) % 360; show()

    def save_and_next(_=None):
        cleaned = clean_text(text_var.get())
        if not cleaned:
            messagebox.showwarning("Teks kosong", "Isi teksnya dulu, atau pakai Lewati."); return
        i = order[state["pos"]]
        df.at[i, "rot"], df.at[i, "text"] = str(state["rot"]), cleaned
        df.to_csv(CSV, index=False); goto(state["pos"] + 1)

    def skip(_=None):
        i = order[state["pos"]]
        df.at[i, "rot"], df.at[i, "text"] = "0", SKIP
        df.to_csv(CSV, index=False); goto(state["pos"] + 1)

    bar = tk.Frame(root); bar.pack(pady=6)
    for label, fn in [("Kembali", lambda: goto(state["pos"] - 1)), ("Putar 90 (Ctrl+R)", rotate),
                      ("Lewati", skip), ("Simpan & Lanjut (Enter)", save_and_next)]:
        tk.Button(bar, text=label, command=fn, width=22).pack(side="left", padx=4)
    root.bind("<Return>", save_and_next); root.bind("<Control-r>", rotate)
    goto(state["pos"]); root.mainloop()

run_label_tool()

In [ ]:
# Cek progres labeling
labeled = load_labeled()
print("terlabel:", len(labeled), "| train:", (labeled.split == "train").sum(), "| val:", (labeled.split == "val").sum())
print("rot:", labeled["rot"].value_counts().to_dict())
labeled[["file", "rot", "text"]].head(10)

## 3. Train YOLO (2 kelas, augmentasi rotasi)
Setelan dari punya temanmu: `yolov8n`, 30 epoch, `degrees=180`, `flipud=0.5`. Train selalu tegak sedangkan test banyak yang miring/terbalik, jadi rotasi diperbanyak lewat augmentasi. Baris `hsv_v` dan `hsv_s` tambahan dari versimu sendiri (test lebih gelap dan tajam menurut EDA), hapus kalau mau persis seperti punya temanmu.
mAP val dihitung dari gambar train yang tegak, jadi terlalu optimistis untuk test. Dilewati otomatis kalau `models/yolo_2cls.pt` sudah ada.

In [ ]:
RETRAIN_YOLO = False     # True = latih ulang (akan MENIMPA models/yolo_2cls.pt)
WEIGHTS = "yolov8n.pt"   # terunduh otomatis sekali; kalau offline, taruh filenya di folder ini

if (MODELS / "yolo_2cls.pt").exists() and not RETRAIN_YOLO:
    print("models/yolo_2cls.pt sudah ada -> training dilewati (set RETRAIN_YOLO = True untuk melatih ulang).")
else:
    yolo_train = YOLO(WEIGHTS)
    yolo_train.train(
        data=str(YOLO_DIR / "data.yaml"), imgsz=IMGSZ, epochs=30, batch=16, workers=0,
        project=str((WORK / "runs").resolve()), name="entete2", exist_ok=True, plots=False, seed=SEED,
        degrees=180, flipud=0.5,        # test banyak yang miring/terbalik
        hsv_v=0.5, hsv_s=0.5,           # test lebih gelap & lebih tajam (dari EDA)
    )
    shutil.copy(Path(yolo_train.trainer.best), MODELS / "yolo_2cls.pt")
    m = YOLO(MODELS / "yolo_2cls.pt").val(data=str(YOLO_DIR / "data.yaml"), imgsz=IMGSZ, workers=0, plots=False)
    print("mAP50:", round(m.box.map50, 4), "| mAP50-95:", round(m.box.map, 4))
    for i, name in enumerate(CLASS_NAMES):
        print(name, "mAP50:", round(m.box.ap50[i], 4))

In [ ]:
# Cek detektor di TEST (tidak punya label): berapa gambar punya kotak entete & seberapa yakin
yolo = YOLO(MODELS / "yolo_2cls.pt")
test_paths = sorted(p for p in (DATA / "test").iterdir() if p.suffix.lower() in {".jpg", ".jpeg", ".png"})
confs = []
for r in yolo.predict([str(p) for p in test_paths], imgsz=IMGSZ, conf=CONF, stream=True, verbose=False):
    c = [float(r.boxes.conf[i]) for i in range(len(r.boxes)) if int(r.boxes.cls[i]) == 0]   # hanya kelas entete
    if c:
        confs.append(max(c))
print(f"test: {len(test_paths)} gambar | ada entete: {len(confs)} ({len(confs)/len(test_paths):.1%})"
      f" | conf median: {np.median(confs):.3f} | min: {min(confs):.3f}")

## 4. Orientation (PaddleOCR, tanpa training)
Dari tiap crop berlabel (yang sudah ditegakkan) dibuat 4 versi putaran, lalu classifier Paddle menebak orientasinya. Karena arah label Paddle bisa berbeda dengan konvensi `rot` di sini, hasilnya dikalibrasi dengan voting menjadi `ORI_MAP` (label Paddle -> derajat cw koreksi) dan disimpan ke `models/orientation_v2.json`. Butuh minimal 20 crop berlabel; makin banyak (dan makin teliti `rot`-nya) makin bisa dipercaya.

In [ ]:
from collections import Counter, defaultdict

lab = load_labeled()
assert len(lab) >= 20, f"Baru {len(lab)} crop berlabel - lengkapi dulu di bagian 2."

votes = defaultdict(Counter)           # votes[label_paddle][rotasi_koreksi_yang_benar]
for f, r in tqdm(list(zip(lab.file, lab.rot)), desc="kalibrasi orientation"):
    upright = load_upright(f, r)
    for applied in (0, 90, 180, 270):
        votes[ori_label(rotate_cw(upright, applied))][needed_rot(applied)] += 1

ORI_MAP = {p: c.most_common(1)[0][0] for p, c in votes.items()}
total = sum(sum(c.values()) for c in votes.values())
acc = sum(c[ORI_MAP[p]] for p, c in votes.items()) / total
for p in sorted(votes):
    print(f"label Paddle {p:3d} -> putar cw {ORI_MAP[p]:3d} | sebaran: {dict(votes[p])}")
print(f"akurasi orientation (setelah kalibrasi): {acc:.3f} dari {total} sampel")

(MODELS / "orientation_v2.json").write_text(json.dumps({str(k): int(v) for k, v in ORI_MAP.items()}))
print("tersimpan -> models/orientation_v2.json")

## 5. Ukur PaddleOCR (CER)
PaddleOCR dipakai **apa adanya**, tidak dilatih, jadi semua crop berlabel boleh dipakai sebagai alat ukur. Crop dibaca dengan `rot` dari labelmu, jadi angka ini hanya mengukur kualitas OCR (bukan kegagalan YOLO atau orientation di test). Karena label sekarang berformat teks penuh, CER-nya sebanding langsung dengan keluaran Paddle. Cell ini juga memilih `ENHANCE` yang CER-nya lebih kecil.

In [ ]:
lab = load_labeled()
assert len(lab) >= 20, f"Baru {len(lab)} crop berlabel - lengkapi dulu di bagian 2."
refs = list(lab.text)

preds_by = {}
for flag in (True, False):
    preds_by[flag] = [paddle_read(load_upright(f, r), use_enhance=flag) for f, r in zip(lab.file, lab.rot)]
    print(f"PaddleOCR (enhance={flag!s:5}) CER: {cer(preds_by[flag], refs):.4f}   ({len(lab)} crop)")

ENHANCE = cer(preds_by[True], refs) <= cer(preds_by[False], refs)
print("dipakai di inference: ENHANCE =", ENHANCE)

# Lihat dengan mata, bukan cuma angka CER
for p, r in list(zip(preds_by[ENHANCE], refs))[:10]:
    print("OK " if p == r else "XX ", "benar:", r, "\n    tebak:", p)

## 6. Inference -> `submission_v2.csv` dan `debug_v2.csv`
Bagian ini mandiri: butuh `models/yolo_2cls.pt`, `models/orientation_v2.json`, dan fungsi dari bagian 0. Dari tiap gambar test diambil **satu kotak `entete` dengan conf tertinggi**.

- `LIMIT = 200` untuk cek cepat (mengikuti 200 baris pertama `sample_submission.csv`); `None` untuk semua 674 gambar. **`submission_v2.csv` hanya valid kalau `LIMIT = None`.**
- Hasil per gambar disimpan di cache, jadi kalau terhenti atau dijalankan ulang yang sudah selesai tidak diulang. Cache otomatis dibuang kalau model atau pengaturan berubah.
- `debug_v2.csv` berisi jumlah kotak, conf, kotak, label orientation, rotasi, dan teks per gambar, untuk analisis di Excel. Crop yang sudah ditegakkan ada di `work/test_crops_v2/`.
- Baris tanpa hasil diisi `0` karena Kaggle menolak sel kosong.

> **Kaggle (internet mati):** paket `paddlepaddle` + `paddleocr` dan model PaddleOCR (deteksi, recognisi, dan `PP-LCNet_x1_0_doc_ori`; semuanya di cache `~/.paddlex/official_models`) harus di-upload sebagai dataset, lalu path-nya diatur, dan `models/yolo_2cls.pt` serta `models/orientation_v2.json` ikut di-upload.

In [ ]:
TEST_DIR, SAMPLE_SUB = DATA / "test", DATA / "sample_submission.csv"
TEST_CROPS = WORK / "test_crops_v2"
LIMIT = None          # None = semua 674 gambar; 200 = cek cepat
FALLBACK = "0"        # Kaggle menolak sel kosong (null)

assert (MODELS / "orientation_v2.json").exists(), "models/orientation_v2.json belum ada - jalankan bagian 4 dulu."
ORI_MAP = {int(k): v for k, v in json.loads((MODELS / "orientation_v2.json").read_text()).items()}
yolo = YOLO(MODELS / "yolo_2cls.pt")
paddle = get_paddle(); get_ori()
print("ENHANCE:", ENHANCE, "| ORI_MAP:", ORI_MAP)

def best_entete(r):
    """Indeks kotak kelas entete dengan conf tertinggi (None kalau tidak ada) dan jumlah kotak entete."""
    best, n = None, 0
    for i in range(len(r.boxes)):
        if int(r.boxes.cls[i]) == 0:
            n += 1
            if best is None or float(r.boxes.conf[i]) > float(r.boxes.conf[best]):
                best = i
    return best, n

def process(name):
    rec = {"id": name, "n_boxes": 0, "detected": 0, "conf": None, "box": None,
           "ori_label": None, "rot": None, "text": ""}
    r = yolo.predict(source=str(TEST_DIR / name), imgsz=IMGSZ, conf=CONF, verbose=False)[0]
    best, rec["n_boxes"] = best_entete(r)
    if best is None:
        return rec
    x1, y1, x2, y2 = [int(v) for v in r.boxes.xyxy[best].tolist()]
    crop = r.orig_img[max(0, y1 - PAD): y2 + PAD, max(0, x1 - PAD): x2 + PAD]
    if crop.size == 0:
        return rec
    lab = ori_label(crop)
    rot = ORI_MAP.get(lab, 0)
    up = rotate_cw(crop, rot)
    cv2.imwrite(str(TEST_CROPS / name), up)
    rec.update(detected=1, conf=round(float(r.boxes.conf[best]), 3), box=f"{x1},{y1},{x2},{y2}",
               ori_label=lab, rot=rot, text=paddle_read(up, ocr=paddle))
    return rec

# Cache hasil per gambar. Dibuang otomatis kalau pengaturan atau model berubah.
CACHE = WORK / "infer_cache_v2.pkl"
sig = (ENHANCE, UPSCALE, CONF, PAD,
       (MODELS / "yolo_2cls.pt").stat().st_mtime, (MODELS / "orientation_v2.json").stat().st_mtime)
records = {}
if CACHE.exists():
    saved = pickle.load(open(CACHE, "rb"))
    if saved.get("sig") == sig:
        records = saved["items"]
print("dari cache:", len(records), "gambar")

sub = pd.read_csv(SAMPLE_SUB)
names = list(sub["id"])[:LIMIT]
for n, name in enumerate(tqdm(names)):
    if name not in records:
        records[name] = process(name)
    if n % 25 == 24:
        pickle.dump({"sig": sig, "items": records}, open(CACHE, "wb"))
pickle.dump({"sig": sig, "items": records}, open(CACHE, "wb"))

dbg = pd.DataFrame([records[n] for n in names])
dbg["text_empty"] = (dbg["text"] == "").astype(int)
dbg.to_csv("debug_v2.csv", index=False)

sub["text"] = [records[n]["text"] if n in records else "" for n in sub["id"]]
n_empty = int((sub["text"] == "").sum())
sub["text"] = sub["text"].replace("", FALLBACK)
sub.to_csv("submission_v2.csv", index=False)

chk = pd.read_csv("submission_v2.csv")                      # baca ulang seperti yang dilakukan Kaggle
assert chk["text"].notna().all(), "masih ada null"
assert sub["text"].str.fullmatch(r"[A-Z0-9/\-]+").all(), "ada karakter terlarang"
print("terdeteksi:", int(dbg.detected.sum()), "/", len(dbg), "| teks kosong (diisi fallback):", n_empty, "dari", len(sub))
print(dbg.groupby("rot", dropna=False).size().to_dict())
dbg.head()